In [39]:
import torch
import cv2
import numpy as np
from depth_anything_3.api import DepthAnything3

## materials used:

- Depth Anything 3 docs - https://github.com/ByteDance-Seed/Depth-Anything-3/blob/main/docs/API.md

- gsplat docs - https://docs.gsplat.studio/main/

- 3D Gaussian Splatting | 3DGS Implementation from Scratch in PyTorch-Only - https://www.youtube.com/watch?v=ZXmN2gWPHus

- https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/


- From Images to Semantic 3D Gaussian Splatting with Python - https://medium.com/data-science-collective/from-images-to-semantic-3d-gaussian-splatting-with-python-complete-guide-ff9d3d240847

In [40]:
def load_model(model="depth-anything/da3-base"):
    # 1. Setup GPU device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2. Load the model directly from Hugging Face
    # Options: "depth-anything/da3-small", "depth-anything/da3-base", "depth-anything/da3-large"
    model = DepthAnything3.from_pretrained("depth-anything/da3-base")
    model = model.to(device)

    return model, device

print('model loaded')

model loaded


In [41]:
import os

def get_images_paths(image_folder):
    image_paths = []

    for n, entry in enumerate(os.scandir(image_folder)):  
        if entry.is_file():
            image_paths.append(entry.path)
    print(f"from folder - {image_folder} - {n} images")

    return image_paths

In [42]:
def run_model_inference(model, image_paths):
    # 4. Run inference (Returns a dictionary with depth and metadata)
    prediction = model.inference(
        image=image_paths,
        infer_gs=True,
    )

    # Extract raw depth data for your point cloud pipeline
    # Note: V3 outputs raw depth and confidence metadata
    print("running model inferance")

    return prediction

In [43]:
import numpy as np
import open3d as o3d
from PIL import Image

# Create a point cloud from rgb and depth map
### - https://stackoverflow.com/questions/59590200/generate-point-cloud-from-depth-image
### - https://codereview.stackexchange.com/questions/79032/generating-a-3d-point-cloud

In [44]:
def create_point_cloud(depth_map, rgb_image, intrinsics, extrinsics, confidence_map, confidence_thresh=0.5):
    h, w = depth_map.shape

    # 1. Standard pinhole projection math (Creates 2D grids)
    fx, fy = intrinsics[0, 0], intrinsics[1, 1]
    cx, cy = intrinsics[0, 2], intrinsics[1, 2]

    u, v = np.meshgrid(np.arange(w), np.arange(h))

    x = (u - cx) * depth_map / fx
    y = (v - cy) * depth_map / fy
    z = depth_map

    # Grid shape: (H, W, 3)
    points_cam = np.stack([x, y, z], axis=-1)
    
    # Color grid shape: (H, W, 3) normalized between 0.0 and 1.0
    colors_grid = rgb_image.astype(np.float32) / 255.0

    # 2. THE THRESHOLD LOGIC
    # Create a 2D boolean mask where confidence is high AND depth is valid (not 0)
    valid_mask = (confidence_map > confidence_thresh) & (depth_map > 0)

    # 3. Flatten the grids into simple lists using the mask
    # This automatically converts the data from (H, W, 3) to (N, 3)
    points_cam_flat = points_cam[valid_mask]  # Shape: (N, 3)
    colors_flat = colors_grid[valid_mask]      # Shape: (N, 3)

    # 4. Transform from Camera Space to World Space
    # Extrinsics broken into Rotation (3x3) and Translation (3,)
    R = extrinsics[:3, :3]
    t = extrinsics[:3, 3]
    
    # Math correction: Shifting points from camera origin to world origin
    # For a flat list of points (N, 3), the matrix math looks like this:
    points_world = (points_cam_flat - t) @ R

    return points_world, colors_flat


In [45]:
import cv2

def match_resolutions(rgb_image, depth_map):
    """
    Resizes the RGB image to match the exact height and width of the depth map.
    """
    # Get the target dimensions from the depth map grid
    target_height, target_width = depth_map.shape
    
    # OpenCV expects (width, height) for the dsize parameter
    resized_rgb = cv2.resize(rgb_image, (target_width, target_height), interpolation=cv2.INTER_LINEAR)
    
    return resized_rgb

In [46]:
# # 2. Extract arrays
# depth_map = prediction.depth[0]
# confidence_map = prediction.conf[0] 
# intrinsics = prediction.intrinsics[0]
# extrinsics = prediction.extrinsics[0]

# # 3. Load the raw image as a numpy array
# rgb_image = np.array(image)

# # 4. Resize the image to match the depth map
# rgb_resized = match_resolutions(rgb_image, depth_map)

# # 5. Call your manual point cloud function safely!
# points_world, colors_flat = create_point_cloud(
#     depth_map=depth_map,
#     rgb_image=rgb_resized, 
#     intrinsics=intrinsics,
#     extrinsics=extrinsics,
#     confidence_map=confidence_map,
#     confidence_thresh=0.5
# )

# print(f"Array shape: {points_world.shape}")

#### link to Vector3dVector - https://www.open3d.org/docs/latest/python_api/open3d.utility.Vector3dVector.html

## Merge multiple point clouds

- medium paper about 'How to Convert 2D Images to 3D Models with Python' -https://medium.com/data-science-collective/depthanything-v2-tutorial-how-to-convert-2d-images-to-3d-models-with-python-2708d295b7e5

In [50]:
def merge_point_clouds(prediction, confidence_thresh=0.5):
    """merge point cloud into one object"""

    print('before')
    all_points=[]
    all_colors=[]

    # number of images in depth map
    frames_count = len(prediction.depth)

    for i in range(frames_count):
        points, colors = create_point_cloud(
            prediction.depth[i],
            prediction.processed_images[i],
            prediction.intrinsics[i],
            prediction.extrinsics[i],
            prediction.conf[i],
            confidence_thresh
        )
        all_points.append(points)
        all_colors.append(colors)

    merged_points = np.vstack(all_points)
    merged_colors = np.vstack(all_colors)

    print(f"Total points in cloud: {len(merged_points)}")
    
    return merged_points, merged_colors

## Statistical outlier remover

### link to docs - https://www.open3d.org/docs/0.9.0/tutorial/Advanced/pointcloud_outlier_removal.html

In [51]:
model, device = load_model()

image_folder = '/home/maciej/dev/depth/car_img'
image_paths = get_images_paths(image_folder)

prediction = run_model_inference(model, image_paths)

merged_points, merged_colors = merge_point_clouds(prediction, confidence_thresh=0.5)

[INFO ] using MLP layer as FFN
from folder - /home/maciej/dev/depth/car_img - 9 images
[WARN ] Images in batch have different sizes [(336, 504), (336, 504), (322, 504), (322, 504), (350, 504), (350, 504), (308, 504), (364, 504), (336, 504), (350, 504)]; center-cropping all to smallest (308,504)
[INFO ] Processed Images Done taking 0.13812470436096191 seconds. Shape:  torch.Size([10, 3, 308, 504])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 22.92898988723755 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0005700588226318359 seconds
running model inferance
before
Total points in cloud: 1552320


In [53]:
# 1. Instantiate an empty Open3D PointCloud object
pcd = o3d.geometry.PointCloud()

# 2. Assign your numpy coordinates (Vector3dVector converts them to Open3D format)
pcd.points = o3d.utility.Vector3dVector((merged_points).astype(np.float32))

# 3. Assign your normalized colors
pcd.colors = o3d.utility.Vector3dVector(merged_colors)

# 4. Save to a standard .ply file
o3d.io.write_point_cloud("first_merged.ply", pcd, write_ascii=False)

print(f"Successfully saved {len(merged_points)}")

Successfully saved 1552320


In [54]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

def visualize_ply_file(file_path):
    pcd = o3d.io.read_point_cloud(file_path)

    pts = np.asarray(pcd.points)
    cols = (np.asarray(pcd.colors) * 255).astype(int)

    # Sample for performance
    idx = np.random.choice(len(pts), 50000, replace=False)

    fig = go.Figure(data=[go.Scatter3d(
        x=pts[idx, 0],
        y=pts[idx, 1],
        z=pts[idx, 2],
        mode='markers',
        marker=dict(
            size=1,
            color=[f'rgb({r},{g},{b})' for r, g, b in cols[idx]],
        )
    )])

    fig.update_layout(scene=dict(aspectmode='data'))
    fig.show()


file_path = ("first_merged.ply")
visualize_ply_file(file_path)
